# Content-Based Recommendation System

## Objective

The objective of this notebook is to build a content-based recommendation system that recommends similar products based on their features such as product title, brand, category, and description.

The recommendation engine uses TF-IDF Vectorization and Cosine Similarity to calculate the similarity between products.

In [18]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


In [19]:
products = pd.read_csv("processed/products_clean.csv")

print("Dataset Loaded Successfully!")

Dataset Loaded Successfully!


In [20]:
print("Shape :", products.shape)

products.head()

Shape : (728, 34)


,s.no,about_item,asin,availability,best_sellers_rank,brand_name,brand_page_url,breadcrumbs,customer_review_summary,default_variant/0,...,rating_distribution/4star,rating_distribution/5star,rating_stars,recent_purchases,scrape_time,seller_name,seller_page_url,title,all_images,rank_1
0,0,Premium Comfort: Crafted from a high-quality c...,B0B59BJG6Y,In Stock,"#56,836 in Clothing, Shoes & Jewelry (See Top ...",MLYENX Store,https://www.amazon.com/stores/MLYENX/page/1FCD...,"Clothing, Shoes & Jewelry › Men › Clothing › A...",Customers find the shirts comfortable and well...,size:Large,...,15%,75%,4.6,50+ bought,03-10-2025 21:42,Greenfive,https://www.amazon.com/gp/help/seller/at-a-gla...,4/5 Pack Mens Polo Shirts Short Sleeve Quick D...,['https://m.media-amazon.com/images/I/41yUF65P...,85.0
1,1,Material: Men's polo shirt is made of soft pol...,B0DLGB4RYH,In Stock,"#50,261 in Clothing, Shoes & Jewelry (See Top ...",COOFANDY Store,https://www.amazon.com/stores/COOFANDY/page/23...,"Clothing, Shoes & Jewelry › Men › Clothing › S...",NaN,size:X-Large,...,17%,66%,4.4,NaN,03-10-2025 21:42,COOFANDY,https://www.amazon.com/gp/help/seller/at-a-gla...,COOFANDY Men's Polo Shirts Short Sleeve Moistu...,['https://m.media-amazon.com/images/I/31a3mSs3...,199.0
2,2,"PERFORMANCE:These men polo shirts are soft,lig...",B0DRXF62JH,In Stock,"#69,641 in Clothing, Shoes & Jewelry (See Top ...",ZITY Store,https://www.amazon.com/stores/ZITY/page/F582B6...,"Clothing, Shoes & Jewelry › Men › Clothing › A...",NaN,size:X-Large,...,37%,63%,4.6,NaN,03-10-2025 21:42,ZITY®,https://www.amazon.com/gp/help/seller/at-a-gla...,ZITY 3 Pack Men Polo Shirts Short Sleeve with ...,['https://m.media-amazon.com/images/I/41J5q-Ua...,111.0
3,3,【Material】: These golf shirts for men are made...,B0DK5FZ325,In Stock,"#194,649 in Clothing, Shoes & Jewelry (See Top...",Rouen Store,https://www.amazon.com/stores/ROUEN/page/4C407...,"Clothing, Shoes & Jewelry › Men › Clothing › A...",NaN,size:Small,...,20%,80%,4.8,NaN,03-10-2025 21:43,MICHEL ROUEN,https://www.amazon.com/gp/help/seller/at-a-gla...,Rouen Mens Golf Shirt Moisture Wicking Dry Fit...,['https://m.media-amazon.com/images/I/31J1ttvf...,266.0
4,4,MOISTURE WICKING: The fabric of the summer gol...,B0BGXTC1FR,In Stock,"#3,664 in Clothing, Shoes & Jewelry (See Top 1...",V VALANCH Store,https://www.amazon.com/stores/VVALANCH/page/0E...,"Clothing, Shoes & Jewelry › Men › Clothing › A...",Customers find the shirt has a good fit and lo...,size:X-Large,...,17%,69%,4.4,NaN,03-10-2025 21:43,Zhengdaqian,https://www.amazon.com/gp/help/seller/at-a-gla...,V VALANCH Mens Polo Shirts Short Sleeve Moistu...,['https://m.media-amazon.com/images/I/31Ff+3Ib...,11.0


In [21]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 728 entries, 0 to 727
Data columns (total 34 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   s.no                       728 non-null    int64  
 1   about_item                 728 non-null    object 
 2   asin                       728 non-null    object 
 3   availability               715 non-null    object 
 4   best_sellers_rank          558 non-null    object 
 5   brand_name                 728 non-null    object 
 6   brand_page_url             649 non-null    object 
 7   breadcrumbs                715 non-null    object 
 8   customer_review_summary    634 non-null    object 
 9   default_variant/0          696 non-null    object 
 10  default_variant/1          658 non-null    object 
 11  default_variant/2          1 non-null      object 
 12  delivery_date              702 non-null    object 
 13  fastest_delivery_date      667 non-null    object 

In [22]:
text_columns = [
    "title",
    "brand_name",
    "breadcrumbs",
    "product_description"
]

for col in text_columns:
    products[col] = products[col].fillna("")

In [23]:
products[text_columns].isnull().sum()

title                  0
brand_name             0
breadcrumbs            0
product_description    0
dtype: int64

In [24]:
products["combined_features"] = (

    products["title"] + " " +

    products["brand_name"] + " " +

    products["breadcrumbs"] + " " +

    products["product_description"]

)

In [25]:
products[
    [
        "title",
        "combined_features"
    ]
].head()

,title,combined_features
0,4/5 Pack Mens Polo Shirts Short Sleeve Quick D...,4/5 Pack Mens Polo Shirts Short Sleeve Quick D...
1,COOFANDY Men's Polo Shirts Short Sleeve Moistu...,COOFANDY Men's Polo Shirts Short Sleeve Moistu...
2,ZITY 3 Pack Men Polo Shirts Short Sleeve with ...,ZITY 3 Pack Men Polo Shirts Short Sleeve with ...
3,Rouen Mens Golf Shirt Moisture Wicking Dry Fit...,Rouen Mens Golf Shirt Moisture Wicking Dry Fit...
4,V VALANCH Mens Polo Shirts Short Sleeve Moistu...,V VALANCH Mens Polo Shirts Short Sleeve Moistu...


In [26]:
tfidf = TfidfVectorizer(
    stop_words="english"
)

In [27]:
tfidf_matrix = tfidf.fit_transform(
    products["combined_features"]
)

In [28]:
print(tfidf_matrix.shape)

(728, 2447)


In [29]:
cosine_sim = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

In [30]:
print(cosine_sim.shape)

(728, 728)


In [31]:
indices = pd.Series(
    products.index,
    index=products["title"]
).drop_duplicates()

In [32]:
indices.head()

title
4/5 Pack Mens Polo Shirts Short Sleeve Quick Dry Moisture Wicking Casual Golf T Shirts for Men                             0
COOFANDY Men's Polo Shirts Short Sleeve Moisture Wicking Golf Shirt Fashion Casual Collared T-Shirt                        1
ZITY 3 Pack Men Polo Shirts Short Sleeve with Pocket Sport Wicking Shirts for Men Casual Athletic Collared T-Shirts        2
Rouen Mens Golf Shirt Moisture Wicking Dry Fit Performance Stripe Casual Collared Short Sleeve Golf Polo Shirts for Men    3
V VALANCH Mens Polo Shirts Short Sleeve Moisture Wicking Golf Polo Athletic Collared Shirt Tennis T-Shirt Tops             4
dtype: int64

##  : Recommendation Function

The recommendation function:

- Finds the selected product
- Calculates similarity scores
- Sorts products based on similarity
- Returns the Top 10 similar products

In [33]:
def recommend_products(product_title, top_n=10):

    if product_title not in indices:
        return "Product not found!"

    idx = indices[product_title]

    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[1:top_n+1]

    product_indices = [i[0] for i in similarity_scores]

    similarity_percent = [
        round(score[1]*100,2)
        for score in similarity_scores
    ]

    recommendations = products.iloc[product_indices][
        [
            "title",
            "brand_name",
            "price_value",
            "rating_stars",
            "rating_count"
        ]
    ].copy()

    recommendations["Similarity (%)"] = similarity_percent

    return recommendations

In [34]:
products["title"].sample(
    10,
    random_state=42
)

39     POLO RALPH LAUREN Men's Classic Fit Cotton Tan...
335    4-7 Pack Long Sleeve Shirts for Men Quick Dry ...
326    Men's Athletic Shorts with Pockets and Elastic...
512    Micoson Womens Tops Dressy Casual Long Sleeve ...
548    PRETTYGARDEN Womens Summer Casual V Neck Boho ...
275    chouyatou Men's Classic Notched Collar Double ...
722    Lee Women's Plus Size Relaxed Fit Straight Leg...
630                                        womens A-line
109    Wrangler Men's Regular Fit Comfort Flex Waist ...
346    Fruit of the Loom Men's Coolzone Boxer Briefs,...
Name: title, dtype: object

In [35]:
sample_product = products.iloc[39]["title"]

print(sample_product)

POLO RALPH LAUREN Men's Classic Fit Cotton Tanks 3-pack


In [36]:
recommend_products(sample_product)

,title,brand_name,price_value,rating_stars,rating_count,Similarity (%)
37,Polo Ralph Lauren Men's Classic Fit Jersey Sho...,POLO RALPH LAUREN Store,55.0000,4.5,0,66.66
26,Polo Ralph Lauren Men's Slim Fit Cotton V-neck...,POLO RALPH LAUREN Store,49.5000,4.5,0,66.44
28,Polo Ralph Lauren Men's Jersey Short Sleeve Tee,POLO RALPH LAUREN Store,42.0000,4.3,0,65.83
16,POLO RALPH LAUREN Men's Slim Fit Stretch Crew Tee,POLO RALPH LAUREN Store,49.5000,4.6,0,65.72
6,POLO RALPH LAUREN Men's Classic Fit Cotton Cre...,POLO RALPH LAUREN Store,49.5000,4.5,0,65.35
101,POLO RALPH LAUREN Men's Slim Fit Cotton Crew U...,POLO Store,49.5000,4.6,0,60.65
8,POLO RALPH LAUREN Men's Classic Fit Cotton V-N...,POLO Store,49.5000,4.5,0,60.62
46,POLO RALPH LAUREN mens Classic Fit Undershirt ...,POLO RALPH LAUREN Store,69.5000,4.6,0,59.54
24,POLO RALPH LAUREN Boys' Multi-Pack Short Sleev...,POLO RALPH LAUREN Store,40.0000,4.0,0,57.05
17,POLO RALPH LAUREN Slim Fit Undershirt w/Wickin...,POLO RALPH LAUREN Store,61.0012,4.6,0,56.01


In [1]:
import pickle
import os

os.makedirs("models", exist_ok=True)

In [1]:
%whos

Interactive namespace is empty.


## Conclusion

The Content-Based Recommendation System successfully recommends products by comparing textual features such as product title, brand, category, and description.

TF-IDF Vectorization converts product metadata into numerical vectors, while Cosine Similarity identifies the most similar products.

This notebook serves as the recommendation engine for the Streamlit application that will be developed later in the project.